# Read TCR units for a clonotype

Read TCR unit names, sequences, abundances and the HMMER logo associated with the selected clonotype.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, SVG, display

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TCRTools").exists() and not (NOTEBOOK_DIR / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "TCRTools"

DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
TCR_UNITS_TABLE = DATA_DIR / "tcr_units_example.csv"
CLONOTYPE_ID = "clone_001"
TOP_N_PER_CHAIN = 1

In [ ]:
def clean_sequence(sequence):
    return "".join(str(sequence).upper().replace("-", "").split())


def read_tcr_units(path):
    units = pd.read_csv(path)
    required = {"clonotype_id", "tcr_unit_name", "chain_type", "sequence_id", "sequence", "abundance"}
    missing = required.difference(units.columns)
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(sorted(missing))}")
    units = units.copy()
    units["chain_type"] = units["chain_type"].str.lower()
    units["sequence"] = units["sequence"].map(clean_sequence)
    units["abundance"] = pd.to_numeric(units["abundance"], errors="coerce").fillna(0)
    return units


def select_clonotype_units(units, clonotype_id):
    matches = units.loc[units["clonotype_id"].astype(str) == str(clonotype_id)].copy()
    if matches.empty:
        available = ", ".join(units["clonotype_id"].astype(str).drop_duplicates().head(10))
        raise ValueError(f"Clonotype '{clonotype_id}' not found. First available IDs: {available}")
    return matches.sort_values("abundance", ascending=False)


def select_most_abundant_units(units, top_n_per_chain=1):
    return (
        units.sort_values("abundance", ascending=False)
        .groupby("chain_type", as_index=False, group_keys=False)
        .head(top_n_per_chain)
        .reset_index(drop=True)
    )

In [ ]:
units = read_tcr_units(TCR_UNITS_TABLE)
units.sort_values(["clonotype_id", "chain_type", "abundance"], ascending=[True, True, False])

In [ ]:
clonotype_units = select_clonotype_units(units, CLONOTYPE_ID)
clonotype_units[["clonotype_id", "tcr_unit_name", "chain_type", "sequence_id", "abundance", "highlight_label", "highlight_sequence"]]

In [ ]:
logo_values = clonotype_units["hmmer_logo"].dropna().astype(str)
logo_path = None
if not logo_values.empty and logo_values.iloc[0].strip():
    logo_path = Path(logo_values.iloc[0])
    if not logo_path.is_absolute():
        logo_path = NOTEBOOK_DIR / logo_path

if logo_path and logo_path.exists():
    if logo_path.suffix.lower() == ".svg":
        display(SVG(filename=str(logo_path)))
    else:
        display(Image(filename=str(logo_path)))
else:
    print("No HMMER logo found for this clonotype")

In [ ]:
selected_units = select_most_abundant_units(clonotype_units, top_n_per_chain=TOP_N_PER_CHAIN)
selected_units.to_csv(OUTPUT_DIR / "selected_tcr_units.csv", index=False)
selected_units[["tcr_unit_name", "chain_type", "sequence_id", "abundance", "highlight_sequence"]]